In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

print("Loading datasets...")
# Adjust the paths if your files are located directly in the main directory instead of a 'dataset' folder
train = pd.read_csv('/dataset/train.csv')
test = pd.read_csv('/dataset/test.csv')

def preprocess_data(df):
    """Handles feature engineering and cleaning"""
    df = df.copy()
    
    # 1. Process the 'timestamp' column (format is usually "HH:MM")
    # We split this into separate 'hour' and 'minute' numeric columns
    df['hour'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else -1)
    df['minute'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[1]) if pd.notnull(x) else -1)
    df = df.drop('timestamp', axis=1)
    
    return df

print("Preprocessing data...")
train_processed = preprocess_data(train)
test_processed = preprocess_data(test)

# Separate features (X) and target (y)
X_train = train_processed.drop(['Index', 'demand'], axis=1)
y_train = train_processed['demand']
X_test = test_processed.drop(['Index'], axis=1)

# 2. Handle Categorical Columns
cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# Fill missing categorical values with a placeholder so the encoder can process them
for col in cat_cols:
    X_train[col] = X_train[col].fillna('Missing')
    X_test[col] = X_test[col].fillna('Missing')

# Use OrdinalEncoder to convert text categories into numbers
# handle_unknown='use_encoded_value' ensures that if the test set has a category not seen in training, it doesn't crash
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

# Ensure numeric columns are strictly float/int (LightGBM requirement)
numeric_cols = ['day', 'NumberofLanes', 'Temperature', 'hour', 'minute']
for col in numeric_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

print("Training LightGBM model (this might take a moment)...")
# Initialize the model. These hyperparameters are a solid starting point for tabular regression
model = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

# Train the model
model.fit(X_train, y_train)

print("Generating predictions...")
# Predict on the test set
predictions = model.predict(X_test)

# 3. Create the Submission File
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': predictions
})

# Save to CSV
submission_filename = '/dataset/submission.csv'
submission.to_csv(submission_filename, index=False)
print(f"Success! Predictions saved to '{submission_filename}'. You can now download this file and submit it to HackerEarth.")